# CatBoost Deep Push — Target 0.683+

CatBoost scored 0.678 public (rank 9) with **untuned defaults**.
This notebook: 100-trial Optuna tune, 10-seed averaging, greedy ensemble.

## Google Colab

1. **Runtime → Change runtime type → GPU** (TPU is not supported for CatBoost / LightGBM / XGBoost).
2. **Run the first code cell** — it clones the GitHub repo and `pip install`s the package (same flow as `01_train_predict_submit`).
3. **Data** — you do **not** need to re-upload CSVs every session if you use Drive:
   - Put `Train.csv`, `Test.csv`, `SampleSubmission.csv` in `My Drive/indabax_data/`.
   - In the **data prep** cell set `DATA_SOURCE = 'drive'` (recommended).
   - Or `DATA_SOURCE = 'upload'` for a one-off file-picker upload.

## Local

Run from the repo (or `notebooks/`). Put the three CSVs in `data/raw/`.

### Optional environment variables

- `FORCE_CPU=1` — force CPU.
- `OPTUNA_TRIALS=20` — quick test (default 100).
- `CATBOOST_GPU_DEVICES=0` — CatBoost GPU id(s).

Colab **GPU** is the practical choice for 100 Optuna trials.

In [ ]:
# ── Colab: clone repo + install. Local: cd to project root. ─────────
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/TinevimboMusingadi/indabax_zimbabwe_hackathon.git'
    REPO_DIR = 'indabax_zimbabwe_hackathon'
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
        !git pull origin main
    else:
        !git clone {REPO_URL}
        os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    !pip install -q -e .
else:
    _root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir(_root)

print(f'Project root: {os.getcwd()}')
print(f'Colab: {IN_COLAB}')


In [ ]:
import os
import subprocess
import sys
import time
import warnings

warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

!{sys.executable} -m pip install -q -U lightgbm xgboost catboost scipy \
    scikit-learn pandas numpy optuna pydantic pyyaml pyarrow
if not IN_COLAB:
    !{sys.executable} -m pip install -q -e .

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
import optuna
import xgboost as xgb
from lightgbm import LGBMClassifier, early_stopping
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.data.loader import ID_COL, TARGET_COL, load_sample_submission, load_test, load_train
from src.features.base import BaseFeatureEngineer
from src.features.encoders.ordinal import OrdinalEncoder
from src.utils.seeding import seed_everything

seed_everything(42)
SEED = 42
N_FOLDS = 5

FORCE_CPU = os.environ.get('FORCE_CPU', '').strip().lower() in ('1', 'true', 'yes')
N_OPTUNA_TRIALS = int(os.environ.get('OPTUNA_TRIALS', '100'))
CATBOOST_GPU_DEVICES = os.environ.get('CATBOOST_GPU_DEVICES', '0').strip() or '0'


def _cuda_visible():
    if FORCE_CPU:
        return False
    try:
        import torch
        return bool(torch.cuda.is_available())
    except ImportError:
        pass
    try:
        r = subprocess.run(
            ['nvidia-smi', '-L'],
            capture_output=True,
            timeout=5,
            check=False,
        )
        return r.returncode == 0 and r.stdout
    except (FileNotFoundError, subprocess.SubprocessError):
        return False


def _catboost_gpu_ok():
    '''Probe CatBoost GPU once; avoids a long Optuna crash on bad drivers.'''
    try:
        x = np.random.default_rng(0).random((120, 4), dtype=np.float32)
        y = (x[:, 0] + x[:, 1] > 1.0).astype(np.int32)
        m = CatBoostClassifier(
            iterations=2,
            depth=3,
            task_type='GPU',
            devices=CATBOOST_GPU_DEVICES,
            verbose=0,
        )
        m.fit(x, y)
        return True
    except Exception as e:
        print(f'CatBoost GPU probe failed ({e!r}) — using CPU for CatBoost.')
        return False


USE_CUDA_VISIBLE = _cuda_visible()
USE_CATBOOST_GPU = bool(USE_CUDA_VISIBLE) and _catboost_gpu_ok()
USE_XGB_GPU = bool(USE_CUDA_VISIBLE) and not FORCE_CPU

if USE_CATBOOST_GPU:
    CB_DEVICE_KW = {'task_type': 'GPU', 'devices': CATBOOST_GPU_DEVICES}
else:
    CB_DEVICE_KW = {'task_type': 'CPU'}

LGBM_GPU_KW = {}
if USE_CUDA_VISIBLE and not FORCE_CPU:
    try:
        _xm = np.random.default_rng(1).random((80, 4), dtype=np.float32)
        _ym = np.zeros(80, dtype=np.int32)
        _probe = LGBMClassifier(
            n_estimators=2,
            max_depth=3,
            device='gpu',
            verbose=-1,
            force_row_wise=True,
        )
        _probe.fit(_xm, _ym)
        LGBM_GPU_KW = {'device': 'gpu', 'force_row_wise': True}
        print('LightGBM: using GPU.')
    except Exception as e:
        print(f'LightGBM GPU not available ({e!r}) — using CPU.')

print('--- Device summary ---')
print(f'  CUDA visible:        {USE_CUDA_VISIBLE}')
print(f'  CatBoost task_type:  {CB_DEVICE_KW}')
print(f'  XGBoost device:      {"cuda" if USE_XGB_GPU else "cpu"}')
print(f'  Optuna trials:       {N_OPTUNA_TRIALS} (set OPTUNA_TRIALS to change)')


In [ ]:
# ── Colab: put CSVs in data/raw/ (Drive or upload) ───────────────────
from pathlib import Path

RAW = Path('data/raw')
REQUIRED = ('Train.csv', 'Test.csv', 'SampleSubmission.csv')

if IN_COLAB:
    RAW.mkdir(parents=True, exist_ok=True)
    # 'drive' = copy from Google Drive (recommended; one-time setup).
    # 'upload' = file picker (no Drive folder needed).
    DATA_SOURCE = 'drive'
    DRIVE_DATA_DIR = '/content/drive/MyDrive/indabax_data'

    if DATA_SOURCE == 'drive':
        from src.utils.colab import mount_drive
        mount_drive(drive_data_dir=DRIVE_DATA_DIR, raw_dir=str(RAW))
    else:
        from src.utils.colab import upload_files
        upload_files(raw_dir=str(RAW))
else:
    missing = [f for f in REQUIRED if not (RAW / f).exists()]
    if missing:
        raise FileNotFoundError(
            f'Missing in data/raw/: {missing}. Place Train.csv, Test.csv, '
            'SampleSubmission.csv in data/raw/ relative to the project root.'
        )

print('Competition files:')
for f in REQUIRED:
    p = RAW / f
    print(f'  {f}: {p.stat().st_size / 1e6:.2f} MB')


In [ ]:
# ── Data + Features (same as 03_competition_push) ────────────────────
train_raw = load_train()
test_raw = load_test()
sample_sub = load_sample_submission()

fe = BaseFeatureEngineer()
y = train_raw[TARGET_COL].copy()
train_fe = fe.fit_transform(train_raw)
test_fe = fe.transform(test_raw)

def _add_competition_features(train_df, test_df):
    cat_cols = [c for c in train_df.columns
                if train_df[c].dtype.name in ('category', 'object')]
    for col in cat_cols:
        freq_map = train_df[col].astype(str).value_counts(normalize=True).to_dict()
        for df in [train_df, test_df]:
            df[f'{col}_freq'] = df[col].astype(str).map(freq_map).fillna(0).astype(np.float32)

    for df in [train_df, test_df]:
        if 'province' in df.columns:
            df['is_urban'] = df['province'].astype(str).isin(['Harare', 'Bulawayo']).astype(np.float32)
        rate = df.get('annual_rate_pct')
        amount = df.get('amount_usd')
        income = df.get('monthly_income_usd')
        term = df.get('term_months')
        age = df.get('client_age_at_approval')
        if rate is not None and amount is not None:
            df['interest_burden'] = (rate * amount / 100).astype(np.float32)
        if rate is not None and income is not None:
            df['rate_income_ratio'] = (rate / income.clip(lower=1)).astype(np.float32)
        if amount is not None and rate is not None and term is not None and income is not None:
            total_cost = amount * (1 + rate / 100 * term.clip(lower=1) / 12)
            df['months_to_repay'] = (total_cost / income.clip(lower=1)).astype(np.float32)
        if age is not None:
            df['age_young'] = (age < 25).astype(np.float32)
            df['age_senior'] = (age > 55).astype(np.float32)
            if income is not None:
                df['age_income_ratio'] = (age / income.clip(lower=1)).astype(np.float32)
        if 'is_mfi_loan' in df.columns and 'has_collateral' in df.columns:
            df['mfi_no_collateral'] = ((df['is_mfi_loan'] == 1) & (df['has_collateral'] == 0)).astype(np.float32)
    return train_df, test_df

train_fe, test_fe = _add_competition_features(train_fe, test_fe)
yv = y.values

# ── CatBoost data frames ────────────────────────────────────────────
cat_col_names = [c for c in train_fe.columns
                 if train_fe[c].dtype.name in ('category', 'object')]

def _catboost_frame(df, feat_cols):
    out = df[feat_cols].copy()
    for c in out.columns:
        if out[c].dtype.name in ('category', 'object'):
            out[c] = out[c].astype(str).fillna('__NA__')
        else:
            out[c] = out[c].astype(np.float32).fillna(0)
    return out

cb_feat_cols = [c for c in train_fe.columns
                if c not in {ID_COL, TARGET_COL}
                and (train_fe[c].dtype.kind in ('i', 'f', 'u')
                     or train_fe[c].dtype.name in ('category', 'object'))]
X_cb_train = _catboost_frame(train_fe, cb_feat_cols)
X_cb_test = _catboost_frame(test_fe, cb_feat_cols)
cb_cat_idx = [i for i, c in enumerate(cb_feat_cols) if c in cat_col_names]

# ── V2 ordinal (for LGBM/XGB in greedy ensemble) ────────────────────
def get_Xy(df, exclude_cols=None):
    exclude = {ID_COL, TARGET_COL} | (exclude_cols or set())
    feat_cols = [c for c in df.columns
                 if c not in exclude and df[c].dtype.kind in ('i', 'f', 'u')]
    X = df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    return X, feat_cols

ord_enc = OrdinalEncoder()
ord_enc.fit(train_fe)
tr_v2 = ord_enc.transform(train_fe)
te_v2 = ord_enc.transform(test_fe)
X_v2, feat_v2 = get_Xy(tr_v2)
Xt_v2, _ = get_Xy(te_v2)

print(f'Train: {train_fe.shape}')
print(f'CatBoost features: {len(cb_feat_cols)} ({len(cb_cat_idx)} categorical)')
print(f'V2 features: {X_v2.shape[1]}')

Train: (38932, 73)
CatBoost features: 71 (12 categorical)
V2 features: 71


## Phase 1: CatBoost Optuna Tuning (100 trials, 3-fold)

Previous CatBoost used defaults: depth=6, lr=0.05, l2_leaf_reg=3.
Search space emphasizes regularization (OOF–public gap ≈ 0.012).

**Speed:** With **GPU** (Colab T4 / local CUDA), this phase is far faster than on CPU.
Trial count: `N_OPTUNA_TRIALS` from env (default 100); use `OPTUNA_TRIALS=20` for a quick test.

In [ ]:
# ── Phase 1: CatBoost Optuna Tuning ──────────────────────────────────
N_TUNE_FOLDS = 3
skf_tune = StratifiedKFold(n_splits=N_TUNE_FOLDS, shuffle=True, random_state=SEED)


def catboost_objective(trial):
    params = {
        'iterations': 5000,
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 3.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
        'one_hot_max_size': trial.suggest_int('one_hot_max_size', 2, 25),
        'grow_policy': trial.suggest_categorical(
            'grow_policy', ['SymmetricTree', 'Lossguide']),
        'eval_metric': 'AUC',
        'random_seed': SEED,
        'verbose': 0,
        'early_stopping_rounds': 150,
        **CB_DEVICE_KW,
    }
    if params['grow_policy'] == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 64)

    oof = np.zeros(len(yv))
    for ti, vi in skf_tune.split(np.zeros(len(yv)), yv):
        pool_tr = Pool(X_cb_train.iloc[ti], yv[ti], cat_features=cb_cat_idx)
        pool_va = Pool(X_cb_train.iloc[vi], yv[vi], cat_features=cb_cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(pool_tr, eval_set=pool_va)
        oof[vi] = m.predict_proba(X_cb_train.iloc[vi])[:, 1]

    return roc_auc_score(yv, oof)


if USE_CATBOOST_GPU:
    print('Starting CatBoost Optuna tuning on GPU...')
    print('Expect ~30–90 min for 100 trials on a T4-class GPU (vs many hours on CPU).')
else:
    print('Starting CatBoost Optuna tuning on CPU (slow for 100 trials).')
    print('Use Colab Runtime → GPU for a large speedup.')

print(f'Trials: {N_OPTUNA_TRIALS}')
t0 = time.time()

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.NopPruner(),
)
study.optimize(catboost_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} minutes')
print(f'Best CatBoost OOF AUC: {study.best_value:.6f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

best_cb_params = study.best_params

Starting CatBoost Optuna tuning (100 trials, 3-fold)...
This will take ~2-3 hours on CPU. Go grab some food.


  0%|          | 0/100 [00:00<?, ?it/s]

## Phase 2: Multi-Seed CatBoost (10 seeds x 5 folds)

Train 10 CatBoost models with best Optuna params but different seeds.
Average their predictions to reduce variance (~0.001-0.002 gain on public LB).

In [ ]:
# ── Phase 2: Multi-Seed CatBoost ─────────────────────────────────────
N_SEEDS = 10
skf5 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def _build_cb_params(seed):
    """Build full CatBoost params from Optuna best + a specific seed."""
    p = {
        'iterations': 5000,
        'depth': best_cb_params['depth'],
        'learning_rate': best_cb_params['learning_rate'],
        'l2_leaf_reg': best_cb_params['l2_leaf_reg'],
        'random_strength': best_cb_params['random_strength'],
        'bagging_temperature': best_cb_params['bagging_temperature'],
        'border_count': best_cb_params['border_count'],
        'min_data_in_leaf': best_cb_params['min_data_in_leaf'],
        'one_hot_max_size': best_cb_params['one_hot_max_size'],
        'grow_policy': best_cb_params['grow_policy'],
        'eval_metric': 'AUC',
        'random_seed': seed,
        'verbose': 0,
        'early_stopping_rounds': 200,
        **CB_DEVICE_KW,
    }
    if p['grow_policy'] == 'Lossguide' and 'max_leaves' in best_cb_params:
        p['max_leaves'] = best_cb_params['max_leaves']
    return p


print(f'Training {N_SEEDS} CatBoost models x {N_FOLDS} folds '
      f"({'GPU' if USE_CATBOOST_GPU else 'CPU'})...")
t0 = time.time()

all_cb_oofs = []
all_cb_tests = []

for seed_i in range(N_SEEDS):
    params = _build_cb_params(seed=seed_i)
    oof = np.zeros(len(yv))
    test_avg = np.zeros(len(X_cb_test))

    for ti, vi in skf5.split(np.zeros(len(yv)), yv):
        pool_tr = Pool(X_cb_train.iloc[ti], yv[ti], cat_features=cb_cat_idx)
        pool_va = Pool(X_cb_train.iloc[vi], yv[vi], cat_features=cb_cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(pool_tr, eval_set=pool_va)
        oof[vi] = m.predict_proba(X_cb_train.iloc[vi])[:, 1]
        test_avg += m.predict_proba(X_cb_test)[:, 1] / N_FOLDS

    auc = roc_auc_score(yv, oof)
    all_cb_oofs.append(oof)
    all_cb_tests.append(test_avg)
    print(f'  Seed {seed_i}: OOF AUC = {auc:.6f}')

# Average across seeds
cb_multi_oof = np.mean(all_cb_oofs, axis=0)
cb_multi_test = np.mean(all_cb_tests, axis=0)
print(f'\nMulti-seed CatBoost ({N_SEEDS} seeds) OOF AUC: '
      f'{roc_auc_score(yv, cb_multi_oof):.6f}')
print(f'Best single-seed CatBoost OOF AUC: '
      f'{max(roc_auc_score(yv, o) for o in all_cb_oofs):.6f}')
print(f'Done in {(time.time()-t0)/60:.1f} minutes')

best_single_idx = int(np.argmax([roc_auc_score(yv, o) for o in all_cb_oofs]))
cb_best_single_test = all_cb_tests[best_single_idx]

## Phase 3: Greedy Ensemble

Train LGBM v2 and XGB v2 with best params from previous tuning.
Then greedily add models to the multi-seed CatBoost base only if they improve OOF AUC.

In [ ]:
# ── Phase 3: Train LGBM + XGB, then greedy ensemble ──────────────────

# Best params from 03_competition_push Optuna runs
lgbm_params = {
    'num_leaves': 26, 'learning_rate': 0.0254, 'min_child_samples': 31,
    'subsample': 0.598, 'colsample_bytree': 0.302,
    'reg_alpha': 1.166, 'reg_lambda': 4.1e-05,
    'min_split_gain': 0.821, 'max_depth': 2,
    'n_estimators': 5000, 'is_unbalance': True,
    'random_state': SEED, 'verbose': -1, 'metric': 'auc',
    **LGBM_GPU_KW,
}

xgb_params = {
    'max_depth': 3, 'learning_rate': 0.0148, 'min_child_weight': 18,
    'subsample': 0.804, 'colsample_bytree': 0.542,
    'reg_alpha': 3.9e-06, 'reg_lambda': 0.0131, 'gamma': 2.593,
    'n_estimators': 5000,
    'scale_pos_weight': float((yv == 0).sum() / (yv == 1).sum()),
    'tree_method': 'hist',
    'eval_metric': 'auc',
    'random_state': SEED, 'verbosity': 0, 'early_stopping_rounds': 200,
}
if USE_XGB_GPU:
    xgb_params['device'] = 'cuda'

# Train LGBM v2
print('Training LGBM v2...')
oof_lgbm = np.zeros(len(yv))
test_lgbm = np.zeros(len(Xt_v2))
for ti, vi in skf5.split(X_v2, yv):
    m = LGBMClassifier(**lgbm_params)
    m.fit(X_v2[ti], yv[ti], eval_set=[(X_v2[vi], yv[vi])],
          callbacks=[early_stopping(200, verbose=False)])
    oof_lgbm[vi] = m.predict_proba(X_v2[vi])[:, 1]
    test_lgbm += m.predict_proba(Xt_v2)[:, 1] / N_FOLDS
print(f'  LGBM v2 OOF AUC: {roc_auc_score(yv, oof_lgbm):.6f}')

# Train XGB v2
print('Training XGB v2...')
oof_xgb = np.zeros(len(yv))
test_xgb = np.zeros(len(Xt_v2))
for ti, vi in skf5.split(X_v2, yv):
    m = xgb.XGBClassifier(**xgb_params)
    m.fit(X_v2[ti], yv[ti], eval_set=[(X_v2[vi], yv[vi])], verbose=False)
    oof_xgb[vi] = m.predict_proba(X_v2[vi])[:, 1]
    test_xgb += m.predict_proba(Xt_v2)[:, 1] / N_FOLDS
print(f'  XGB v2 OOF AUC:  {roc_auc_score(yv, oof_xgb):.6f}')

# ── Greedy forward selection ─────────────────────────────────────────
candidates = {
    'lgbm_v2': (oof_lgbm, test_lgbm),
    'xgb_v2': (oof_xgb, test_xgb),
}

# Start with multi-seed CatBoost
ensemble_oofs = [rankdata(cb_multi_oof)]
ensemble_tests = [rankdata(cb_multi_test)]
ensemble_names = ['catboost_multi']
current_auc = roc_auc_score(yv, cb_multi_oof)
print(f'\nGreedy ensemble — starting AUC: {current_auc:.6f} (catboost_multi)')

remaining = dict(candidates)
improved = True
while improved and remaining:
    improved = False
    best_name = None
    best_auc = current_auc
    best_oof_rank = None
    best_test_rank = None

    for name, (oof_c, test_c) in remaining.items():
        trial_oofs = ensemble_oofs + [rankdata(oof_c)]
        trial_blend = np.mean(trial_oofs, axis=0)
        trial_auc = roc_auc_score(yv, trial_blend)
        if trial_auc > best_auc + 1e-7:
            best_auc = trial_auc
            best_name = name
            best_oof_rank = rankdata(oof_c)
            best_test_rank = rankdata(test_c)

    if best_name:
        ensemble_oofs.append(best_oof_rank)
        ensemble_tests.append(best_test_rank)
        ensemble_names.append(best_name)
        current_auc = best_auc
        del remaining[best_name]
        improved = True
        print(f'  + Added {best_name} -> AUC = {current_auc:.6f}')

print(f'\nFinal greedy ensemble: {ensemble_names}')
print(f'Final greedy OOF AUC: {current_auc:.6f}')

greedy_oof = np.mean(ensemble_oofs, axis=0)
greedy_test = np.mean(ensemble_tests, axis=0)

## Phase 5: Submissions

Three files — submit ALL three and keep the best on public LB.

In [ ]:
# ── Phase 5: Generate submissions ────────────────────────────────────
os.makedirs('submissions', exist_ok=True)

# 1. Multi-seed tuned CatBoost (most likely best on public LB)
sub1 = sample_sub.copy()
sub1[TARGET_COL] = cb_multi_test
sub1.to_csv('submissions/sub_catboost_tuned.csv', index=False)

# 2. Greedy ensemble
sub2 = sample_sub.copy()
sub2[TARGET_COL] = greedy_test
sub2.to_csv('submissions/sub_greedy_ensemble.csv', index=False)

# 3. Best single-seed CatBoost (backup)
sub3 = sample_sub.copy()
sub3[TARGET_COL] = cb_best_single_test
sub3.to_csv('submissions/sub_catboost_single_seed.csv', index=False)

print('=== Submissions generated ===')
for f in ['sub_catboost_tuned.csv', 'sub_greedy_ensemble.csv',
          'sub_catboost_single_seed.csv']:
    s = pd.read_csv(f'submissions/{f}')
    t = s[TARGET_COL]
    print(f'  {f:35s} mean={t.mean():.4f}  min={t.min():.4f}  max={t.max():.4f}')

print('\n>>> Submit ALL THREE — different ones often score differently <<<')
print('>>> on public vs private LB. Keep the best!                  <<<')